In [0]:
from pyspark.sql import functions as F
import uuid


In [0]:
pipeline_run_id = str(uuid.uuid4())

In [0]:
orders_curated_df = spark.table("ecom_medallion_pipeline.silver.orders_curated")

In [0]:
# 3. Apply safety filters before Gold aggregation
gold_base_df = (
    orders_curated_df
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("order_date").isNotNull())
    .filter(F.col("quantity").isNotNull())
    .filter(F.col("unit_price").isNotNull())
    .filter(F.col("order_amount").isNotNull())
)


In [0]:
# Building Gold Table :Daily Product Sales
# Aggregation performed on :one row per order_date, product_id, product_name and category

daily_product_sales_df = (
    gold_base_df
    .groupBy(
        F.col("order_date"),
        F.col("product_id"),
        F.col("product_name"),
        F.col("category")
    )
    .agg(
        F.countDistinct("order_id").cast("int").alias("total_orders"),
        F.sum("quantity").cast("int").alias("total_quantity"),
        F.round(F.sum("order_amount"), 2).cast("double").alias("total_sales")
    )
    .withColumn("pipeline_run_id", F.lit(pipeline_run_id))
)

In [0]:
# Building Gold Table: Daily Product Sales
# Aggregation performed on: one row per order_date + product_id + product_name + category

daily_product_sales_df = (
    gold_base_df
    .groupBy(
        F.col("order_date"),
        F.col("product_id"),
        F.col("product_name"),
        F.col("category")
    )
    .agg(
        F.countDistinct("order_id").cast("int").alias("total_orders"),
        F.sum("quantity").cast("int").alias("total_quantity"),
        F.round(F.sum("order_amount"), 2).cast("double").alias("total_sales")
    )
    .withColumn("pipeline_run_id", F.lit(pipeline_run_id))
)

In [0]:
# Build Gold Table: Category Sales
# Aggregation performed on: one row per order_date + category

category_sales_df = (
    gold_base_df
    .groupBy(
        F.col("order_date"),
        F.col("category")
    )
    .agg(
        F.countDistinct("order_id").cast("int").alias("total_orders"),
        F.sum("quantity").cast("int").alias("total_quantity"),
        F.round(F.sum("order_amount"), 2).cast("double").alias("total_sales")
    )
    .withColumn("pipeline_run_id", F.lit(pipeline_run_id))
)

In [0]:
# Write Gold Daily Product Sales table

daily_product_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecom_medallion_pipeline.gold.daily_product_sales")

# ----------------------------------------------------------
# Write Gold Category Sales table
# ----------------------------------------------------------

category_sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecom_medallion_pipeline.gold.category_sales")

In [0]:
# Data Validation

print("Pipeline Run ID:", pipeline_run_id)

print("Daily Product Sales Count:")
spark.sql("SELECT COUNT(*) AS total_rows FROM ecom_medallion_pipeline.gold.daily_product_sales").show()

print("Category Sales Count:")
spark.sql("SELECT COUNT(*) AS total_rows FROM ecom_medallion_pipeline.gold.category_sales").show()

print("Daily Product Sales Preview:")
display(spark.table("ecom_medallion_pipeline.gold.daily_product_sales"))

print("Category Sales Preview:")
display(spark.table("ecom_medallion_pipeline.gold.category_sales"))

Pipeline Run ID: fe2295cc-18f2-4f6e-abcd-7bf0c642844f
Daily Product Sales Count:
+----------+
|total_rows|
+----------+
|        26|
+----------+

Category Sales Count:
+----------+
|total_rows|
+----------+
|        13|
+----------+

Daily Product Sales Preview:


order_date,product_id,product_name,category,total_orders,total_quantity,total_sales,pipeline_run_id
2026-03-20,P103,Webcam HD,Electronics,1,1,799.0,fe2295cc-18f2-4f6e-abcd-7bf0c642844f
2026-03-21,P111,Desk Lamp,Furniture,1,2,598.0,fe2295cc-18f2-4f6e-abcd-7bf0c642844f
2026-03-20,P100,Wireless Mouse,Accessories,2,5,2995.0,fe2295cc-18f2-4f6e-abcd-7bf0c642844f
2026-03-21,P106,Mouse Pad,Accessories,1,5,495.0,fe2295cc-18f2-4f6e-abcd-7bf0c642844f
2026-03-21,P100,Wireless Mouse,Accessories,2,3,1797.0,fe2295cc-18f2-4f6e-abcd-7bf0c642844f
2026-03-21,P101,Mechanical Keyboard,Accessories,1,1,1299.0,fe2295cc-18f2-4f6e-abcd-7bf0c642844f
2026-03-21,P105,Laptop Stand,Accessories,1,1,2499.0,fe2295cc-18f2-4f6e-abcd-7bf0c642844f
2026-03-20,P999,UNKNOWN,UNKNOWN,1,1,499.0,fe2295cc-18f2-4f6e-abcd-7bf0c642844f
2026-03-21,P109,USB-C Charger,Power,1,1,699.0,fe2295cc-18f2-4f6e-abcd-7bf0c642844f
2026-03-20,P102,USB Hub,Accessories,2,4,796.0,fe2295cc-18f2-4f6e-abcd-7bf0c642844f


Category Sales Preview:


order_date,category,total_orders,total_quantity,total_sales,pipeline_run_id
2026-03-20,Accessories,8,17,10583.0,fe2295cc-18f2-4f6e-abcd-7bf0c642844f
2026-03-21,Storage,2,3,2697.0,fe2295cc-18f2-4f6e-abcd-7bf0c642844f
2026-03-20,UNKNOWN,1,1,499.0,fe2295cc-18f2-4f6e-abcd-7bf0c642844f
2026-03-21,UNKNOWN,1,1,450.0,fe2295cc-18f2-4f6e-abcd-7bf0c642844f
2026-03-21,Power,1,1,699.0,fe2295cc-18f2-4f6e-abcd-7bf0c642844f
2026-03-21,Furniture,3,7,8593.0,fe2295cc-18f2-4f6e-abcd-7bf0c642844f
2026-03-20,Electronics,2,3,3797.0,fe2295cc-18f2-4f6e-abcd-7bf0c642844f
2026-03-21,Accessories,8,17,8883.0,fe2295cc-18f2-4f6e-abcd-7bf0c642844f
2026-03-20,Audio,1,1,3999.0,fe2295cc-18f2-4f6e-abcd-7bf0c642844f
2026-03-21,Audio,2,2,4998.0,fe2295cc-18f2-4f6e-abcd-7bf0c642844f
